In [60]:
import torch
from transformers import  CamembertForMaskedLM, CamembertTokenizer, CamembertModel,RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline
from transformers import BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM 
from transformers import TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss

In [2]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)

In [50]:
#Data
source = "test/Slovak.txt"
file = open(source, "r", encoding = 'utf-8')
lines = file.readlines()

data = []

for l in range(12):
    line = lines[l]
    
    data.append(line)

In [51]:
def bert_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = "[CLS] " + text + " [SEP]" #special token for BERT, RoBERTa
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    length = len(tokenized_text)-2
    for masked_index in range(1,len(tokenized_text)-1):
        # Mask a token that we will try to predict back with `BertForMaskedLM`
        masked_word = tokenized_text[masked_index]
        #tokenized_text[masked_index] = '<mask>' #special token for XLNet
        tokenized_text[masked_index] = '[MASK]' #special token for BERT, RoBerta
        # Convert token to vocabulary indices
        indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
        index = torch.tensor(tokenizer.convert_tokens_to_ids(masked_word))
        tokens_tensor = torch.tensor([indexed_tokens])
        tokens_tensor = tokens_tensor.to('cuda')
        index = index.to('cuda')
        #masked_tensor = torch.tensor([masked_index])
        with torch.no_grad():
            outputs = model(tokens_tensor.to('cuda'))
        prediction_scores = outputs[0]
        prediction_scores = prediction_scores.view(-1, model.config.vocab_size)
        prediction_scores = prediction_scores[masked_index].unsqueeze(0)
        loss_fct = CrossEntropyLoss(ignore_index=-1)  # -1 index = padding token
        masked_lm_loss = loss_fct(prediction_scores, index.view(-1))
        tokenized_text[masked_index] = masked_word
        sentence_score -= masked_lm_loss.item()
        tokenized_text[masked_index] = masked_word
    sentence_score = sentence_score/length
    return sentence_score

In [64]:
def score_model(model, tokenizer, data):
    #opts = ["stały","dzieci","dalszy","trudno","bawi","krajach","pieniądze","kosztowne","młode","wierzą","osiąga","trenowania"]
    opts = ["vyrástli","deti","druhoradé","ťažké","hrá","krajinách","peniaze","drahé","nevyvinuté","veria","dostane","trénovania"]
    for d in data:
        print(d)
        print("Correct option is: ", data.index(d) + 1, opts[data.index(d)])
        for o in opts:
            sentence = d.replace("{}", o)
            score = bert_predict(sentence, model, tokenizer)
            print(o, score)
        print()
        print()
        

In [65]:
#Polish
#print(torch.cuda.is_available())
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
#print(bert_predict("text text text", model, tokenizer))
scoredModel = score_model(model, tokenizer, data)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Rodičia tých detí, ktoré prejavujú zvýšený záujem o niektorý zo športov, majú pred sebou neľahké rozhodovanie. Mali by umožniť deťom trénovať toľko, aby z nich {} špičkoví športovci a športovkyne? 

Correct option is:  1 vyrástli
vyrástli -5.7804600571242855
deti -5.57143211580705
druhoradé -6.017880390371595
ťažké -5.919204410271985
hrá -6.023110420388334
krajinách -5.901382534418787
peniaze -5.935974929588181
drahé -5.940363347098447
nevyvinuté -5.8177312073401275
veria -5.958890397479569
dostane -5.8881915596084315
trénovania -5.725605300095465

Pre mnohé {} to znamená začať od malička.

Correct option is:  2 deti
vyrástli -6.914206577464938
deti -7.896527115787778
druhoradé -8.16789474884669
ťažké -7.897684919834137
hrá -8.239580227205387
krajinách -8.15409681548675
peniaze -7.944473703702291
drahé -7.665995719177382
nevyvinuté -7.343843907117844
veria -7.850756458938122
dostane -7.717996426991054
trénovania -7.9020481715599695

Škola, kamaráti a iné záujmy musia byť {}.

Correct o

In [66]:
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
scoredModel = score_model(model, tokenizer, data)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
All TF 2.0 model weights were used when initializing AlbertForMaskedLM.

Some weights of AlbertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['predictions.decoder.weight', 'predictions.decoder.bias', 'predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Rodičia tých detí, ktoré prejavujú zvýšený záujem o niektorý zo športov, majú pred sebou neľahké rozhodovanie. Mali by umožniť deťom trénovať toľko, aby z nich {} špičkoví športovci a športovkyne? 

Correct option is:  1 vyrástli
vyrástli -5.654819156617334
deti -5.6393411457538605
druhoradé -5.6915464517031165
ťažké -5.695634111963414
hrá -5.701333178103798
krajinách -5.71781106946403
peniaze -5.615148278809077
drahé -5.701872792612347
nevyvinuté -5.450306852322978
veria -5.706075398378993
dostane -5.7168804138071
trénovania -5.528598081370866

Pre mnohé {} to znamená začať od malička.

Correct option is:  2 deti
vyrástli -7.418411550040428
deti -6.266289110367115
druhoradé -5.678533907447543
ťažké -5.269255460905177
hrá -7.06787252984941
krajinách -5.627383413796242
peniaze -5.634705488170896
drahé -6.37349571660161
nevyvinuté -4.4306143040635755
veria -6.317642002724684
dostane -6.887704694022735
trénovania -5.935411174914667

Škola, kamaráti a iné záujmy musia byť {}.

Correct opti

In [67]:
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
scoredModel = score_model(model, tokenizer, data)

Rodičia tých detí, ktoré prejavujú zvýšený záujem o niektorý zo športov, majú pred sebou neľahké rozhodovanie. Mali by umožniť deťom trénovať toľko, aby z nich {} špičkoví športovci a športovkyne? 

Correct option is:  1 vyrástli
vyrástli -1.7316262893010572
deti -2.1701388286315444
druhoradé -2.128870011928038
ťažké -2.323732481482972
hrá -2.4228406276661376
krajinách -2.270622591065167
peniaze -2.244788222071033
drahé -2.2839660181078734
nevyvinuté -2.1904859237601713
veria -2.372100600458926
dostane -2.4940199511552144
trénovania -2.4637716351592824

Pre mnohé {} to znamená začať od malička.

Correct option is:  2 deti
vyrástli -4.564689515701805
deti -3.885125314084101
druhoradé -3.807174656766859
ťažké -5.187848193659995
hrá -5.534049733706257
krajinách -5.139955625609112
peniaze -5.117026750922806
drahé -5.322589709923384
nevyvinuté -4.092767303190684
veria -5.3614903894965265
dostane -5.985280637122581
trénovania -4.639597197645344

Škola, kamaráti a iné záujmy musia byť {}.

Co

In [68]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
scoredModel = score_model(model, tokenizer, data)

Rodičia tých detí, ktoré prejavujú zvýšený záujem o niektorý zo športov, majú pred sebou neľahké rozhodovanie. Mali by umožniť deťom trénovať toľko, aby z nich {} špičkoví športovci a športovkyne? 

Correct option is:  1 vyrástli
vyrástli -14.773028860921444
deti -14.697626953539642
druhoradé -14.621416985988617
ťažké -14.729084989298945
hrá -14.720670249151146
krajinách -14.777027171591055
peniaze -14.69391967939294
drahé -14.754468866016554
nevyvinuté -14.622151310245195
veria -14.745933791865474
dostane -14.741564470788706
trénovania -14.762017797916494

Pre mnohé {} to znamená začať od malička.

Correct option is:  2 deti
vyrástli -12.970593124628067
deti -12.63993501663208
druhoradé -12.77248043484158
ťažké -12.795474886894226
hrá -12.802441358566284
krajinách -12.921967446804047
peniaze -12.7170070707798
drahé -12.846367478370667
nevyvinuté -12.69187503390842
veria -12.771697700023651
dostane -12.797339737415314
trénovania -12.964813288520364

Škola, kamaráti a iné záujmy musia b